<a href="https://colab.research.google.com/github/Viji-codes/Smart---prompt--template/blob/main/Prompt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# =====================================================================
# STEP 1: INSTALL DEPENDENCIES
# =====================================================================
!pip install -q -q google-genai ipywidgets pandas --no-warn-conflicts

import os
import re
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from google import genai
from google.colab import userdata

# =====================================================================
# STEP 2: INITIALIZE GEMINI CLIENT & BASE DATASET
# =====================================================================
# Fetch API key securely from Colab Secrets (Key name: GEMINI_API_KEY)
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

# Initial Prompt Library Collection
library_data = [
    {
        "ID": 1,
        "Category": "Coding",
        "Title": "Code Reviewer",
        "Prompt": "Review the following {language} code for bugs, performance bottlenecks, and best practices:\n\n{code}",
        "Rating": 5,
        "Tags": "python, refactoring, bugs"
    },
    {
        "ID": 2,
        "Category": "Marketing",
        "Title": "Tagline Generator",
        "Prompt": "Generate 5 catchy, memorable taglines for a {product} targeting {audience}.",
        "Rating": 4,
        "Tags": "branding, copy, ads"
    },
    {
        "ID": 3,
        "Category": "Writing",
        "Title": "Tone Adapter",
        "Prompt": "Rewrite the following text into a {tone} tone while preserving key information:\n\n{text}",
        "Rating": 5,
        "Tags": "editing, style, rewrite"
    }
]

df_library = pd.DataFrame(library_data)

# =====================================================================
# STEP 3: HELPER PARSER FUNCTIONS
# =====================================================================
def extract_variables(template_str):
    """Finds all unique {variable} placeholders in a prompt string."""
    return list(dict.fromkeys(re.findall(r'\{([a-zA-Z0-9_]+)\}', template_str)))

def format_prompt(template_str, variable_dict):
    """Replaces variables with user values safely."""
    try:
        return template_str.format(**variable_dict)
    except KeyError as e:
        return f"Error: Missing value for placeholder {e}"

# =====================================================================
# STEP 4: BUILD INTERACTIVE DASHBOARD UI
# =====================================================================
category_select = widgets.Dropdown(
    options=sorted(df_library['Category'].unique().tolist()),
    description='Category:',
    style={'description_width': 'initial'}
)

template_select = widgets.Dropdown(
    description='Template:',
    style={'description_width': 'initial'}
)

variable_box = widgets.VBox()
rating_slider = widgets.IntSlider(value=5, min=1, max=5, description='Rating:')
run_btn = widgets.Button(description='⚡ Run Prompt Test', button_style='primary')
save_btn = widgets.Button(description='💾 Save Updated Rating', button_style='success')
output_display = widgets.Output()

var_inputs = {}

def refresh_templates(*args):
    filtered = df_library[df_library['Category'] == category_select.value]
    template_select.options = filtered['Title'].tolist()
    refresh_variable_fields()

def refresh_variable_fields(*args):
    global var_inputs
    var_inputs = {}
    if not template_select.value:
        return

    selected_row = df_library[df_library['Title'] == template_select.value].iloc[0]
    prompt_text = selected_row['Prompt']
    rating_slider.value = int(selected_row['Rating'])

    vars_found = extract_variables(prompt_text)

    fields = []
    for v in vars_found:
        txt = widgets.Text(description=f"{v}:", placeholder=f"Enter value for {v}")
        var_inputs[v] = txt
        fields.append(txt)

    variable_box.children = fields

category_select.observe(refresh_templates, 'value')
template_select.observe(refresh_variable_fields, 'value')

def execute_prompt(b):
    with output_display:
        clear_output()
        selected_row = df_library[df_library['Title'] == template_select.value].iloc[0]
        template_text = selected_row['Prompt']

        values = {k: widget.value for k, widget in var_inputs.items()}
        compiled_prompt = format_prompt(template_text, values)

        print("================ 📝 EXECUTING PROMPT ================")
        print(compiled_prompt)
        print("====================================================\n")

        try:
            response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=compiled_prompt
            )
            print("================ 🤖 GEMINI RESPONSE ================")
            print(response.text)
            print("====================================================")
        except Exception as e:
            print(f"❌ Error generating response: {e}")

def update_rating(b):
    global df_library
    title = template_select.value
    df_library.loc[df_library['Title'] == title, 'Rating'] = rating_slider.value
    df_library.to_csv('prompt_library.csv', index=False)
    with output_display:
        clear_output()
        print(f"✅ Updated rating for '{title}' to {rating_slider.value}/5 stars and saved to prompt_library.csv!")

run_btn.on_click(execute_prompt)
save_btn.on_click(update_rating)

# Initialize view and display controls
refresh_templates()

print("🚀 SMART PROMPT LIBRARY DASHBOARD")
display(category_select, template_select, widgets.HTML("<hr><b>Fill Template Variables:</b>"))
display(variable_box, rating_slider, widgets.HBox([run_btn, save_btn]), output_display)

🚀 SMART PROMPT LIBRARY DASHBOARD


Dropdown(description='Category:', options=('Coding', 'Marketing', 'Writing'), style=DescriptionStyle(descripti…

Dropdown(description='Template:', options=('Code Reviewer',), style=DescriptionStyle(description_width='initia…

HTML(value='<hr><b>Fill Template Variables:</b>')

IntSlider(value=5, description='Rating:', max=5, min=1)

Output()